# S12 · Pandas, EDA, visualisation and working with APIs

**Your role:** An operations analyst for a fictional retailer. What do *valid Fulfilled transactions* tell us about discount-adjusted order value across its East, Central and West hubs and product categories?

You will first inspect and prepare two weekly CSV batches and an Excel hub lookup. Then you will obtain **separate, real** India population observations from the World Bank Indicators API and make a year-versus-population chart. The two datasets answer different questions; do **not** join them.

**What you will leave with:** a traceable local quality audit, a reconciled hub/category summary, a useful local EDA graphic, a source-labelled API population chart, and short interpretations with limits. The local data are synthetic; the World Bank series is a dated or live external source.

This notebook begins with guided inspection and short checks. Cells marked **TODO** are deliberately unfinished. Complete them in order; *Run All* is expected to stop at an unfinished TODO. The final integrated practice revisits one local check and then focuses on the API pathway.


## 1 · Set up files and packages

Put these **four unchanged files** beside this notebook: `S12_Transactions_Week1.csv`, `S12_Transactions_Week2.csv`, `S12_Hub_Reference.xlsx`, and `S12_WorldBank_Population_Snapshot.json`. The optional `S12_Transaction_Data_Dictionary.md` explains the fields. The snapshot is an offline fallback, not a precompleted API result.

In local Jupyter/JupyterLab, check the notebook's current working directory below; if your notebook server starts from another folder, change `DATA_DIR`. In Colab, upload the four files through the Files pane into the current runtime; uploads disappear after a runtime reset. No GitHub login, API key or paid service is needed. If a package is missing, run `%pip install pandas openpyxl matplotlib seaborn requests` **in a notebook cell in this kernel**, then restart the kernel and run setup again. Avoid installing over an already working environment just to chase the newest versions.


In [ ]:
from pathlib import Path
import importlib.util

needed = ["pandas", "openpyxl", "matplotlib", "seaborn", "requests"]
missing_packages = [name for name in needed if importlib.util.find_spec(name) is None]
if missing_packages:
    raise ModuleNotFoundError(
        f"Missing {missing_packages}. In this notebook kernel run: "
        "%pip install pandas openpyxl matplotlib seaborn requests; then restart."
    )

import json
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import requests
from IPython.display import display

print("Working directory:", Path.cwd())
print("pandas:", pd.__version__, "seaborn:", sns.__version__,
      "matplotlib:", __import__("matplotlib").__version__,
      "requests:", requests.__version__)


In [ ]:
DATA_DIR = Path(".")  # Change this only if the files are not in the working directory.
file_names = [
    "S12_Transactions_Week1.csv",
    "S12_Transactions_Week2.csv",
    "S12_Hub_Reference.xlsx",
    "S12_WorldBank_Population_Snapshot.json",
]
missing_files = [name for name in file_names if not (DATA_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(
        f"Missing {missing_files} in {DATA_DIR.resolve()}. "
        "Upload the files there or change DATA_DIR."
    )
print("All four files found in", DATA_DIR.resolve())


## 2 · Read labelled transaction tables

**Grain:** one synthetic, single-category transaction per row. A DataFrame can keep a text ID, date, hub, category, numeric-looking fields and status together. Its default index locates a row *inside this DataFrame*; `transaction_id` is the business identifier. Repeated hubs or categories are ordinary, not duplicate transactions.

Predict: Which file has the earlier transaction dates? Why might one record in Week 2 still show a Week 1 date? What should a blank price mean when valuing an order?


In [ ]:
week1 = pd.read_csv(DATA_DIR / "S12_Transactions_Week1.csv",
                    dtype={"transaction_id": "string"})
week2 = pd.read_csv(DATA_DIR / "S12_Transactions_Week2.csv",
                    dtype={"transaction_id": "string"})
try:
    hubs = pd.read_excel(DATA_DIR / "S12_Hub_Reference.xlsx",
                         sheet_name="Hub Reference", engine="openpyxl")
except ImportError as exc:
    raise RuntimeError("Excel reader missing. Run %pip install openpyxl in this kernel.") from exc

print("Week 1 shape:", week1.shape, "Week 2 shape:", week2.shape,
      "Hub lookup shape:", hubs.shape)
print("Transaction fields:", week1.columns.tolist())
display(week1.head(3), hubs)


In [ ]:
print("Week 1 dtypes:\n", week1.dtypes)
print("Week 2 dtypes:\n", week2.dtypes)
print("Week 1 missing by field:\n", week1.isna().sum())
print("Week 2 missing by field:\n", week2.isna().sum())
print("Week 1 summary of inferred numeric fields:")
display(week1.describe())
# Uncomment if you want the full non-null count and dtype inspection:
# week1.info()


`week1["units"]` selects one **Series**; `week1[["units"]]` keeps a one-column **DataFrame**. Their shapes differ. A numeric-looking identifier remains a key, never something to average. A summary's non-null count may be smaller than the table's row count.

**Micro-check:** What do the inferred `units` dtypes suggest? Does `describe()` prove that every row is valid?


In [ ]:
one_column = week1["units"]
one_column_table = week1[["units"]]
print(type(one_column).__name__, one_column.shape)
print(type(one_column_table).__name__, one_column_table.shape)
print("First row by position (.iloc):")
display(week1.iloc[[0], :4])


### Short selection practice · during the filtering lesson

Use `.loc` with a Boolean condition to show the **Fulfilled** Week 1 rows for hub `E`, selecting only ID, date, category and status. Then sort that small view by date. Parenthesise each condition before joining them with `&`. This is a source inspection, **not yet** an eligible-value table.

Hint 1: `week1["status"].eq("Fulfilled")` makes one condition. Hint 2: `.loc[condition, ["col1", "col2"]]` selects rows and named columns. What is the difference between `.loc` labels and the `.iloc` position example above?


In [ ]:
# TODO 1: Fill the two expressions, then inspect the resulting rows.
east_fulfilled = None
east_fulfilled_sorted = None
if east_fulfilled is None or east_fulfilled_sorted is None:
    raise NotImplementedError("Complete the .loc filter and .sort_values practice.")
display(east_fulfilled_sorted.head())


## 3 · Find, explain, choose, count, verify

The reporting measure will be `net_order_value = units * unit_price * (1 - discount_rate)`: **discount-adjusted pre-tax Fulfilled order value (USD)**. It excludes tax and shipping and is not recognised accounting revenue. A blank price is missing, not zero. Cancelled orders stay visible in raw records but do not enter this measure.

Before calculating, combine the two same-schema weekly batches **vertically** with `pd.concat`. File membership is not a date field: an exact carried-over Week 1 row in Week 2 retains its earlier date. Check the combined count against the sum of batch counts.


In [ ]:
# TODO 2: Concatenate both batches in file order, resetting the pandas index.
raw = None
if raw is None:
    raise NotImplementedError("Use pd.concat on the two weekly DataFrames.")
assert raw.shape[0] == len(week1) + len(week2)
assert raw.columns.tolist() == week1.columns.tolist() == week2.columns.tolist()
print("Raw rows:", len(raw))


In [ ]:
# An entire repeated record and a repeated business ID answer different questions.
full_repeat = raw.duplicated(subset=week1.columns.tolist(), keep=False)
id_repeat = raw["transaction_id"].duplicated(keep=False)
print("Rows participating in full repeats:", int(full_repeat.sum()))
print("Rows participating in repeated IDs:", int(id_repeat.sum()))
display(raw.loc[full_repeat, week1.columns.tolist()])
display(raw.loc[id_repeat, ["transaction_id", "transaction_date", "hub_code", "category"]])


**Decision:** Is the repeated row a second order or the same transaction copied across files? Keep the first original row and preserve `raw` for audit. Verify the deduplicated working table has unique `transaction_id` values. A repeated category/hub/date alone would not justify deleting a transaction.


In [ ]:
# TODO 3: Make a deduplicated copy; do not overwrite raw.
working = None
if working is None:
    raise NotImplementedError("Inspect then remove the exact repeated row in working.")
print("Raw / working / removed:", len(raw), len(working), len(raw) - len(working))
assert working["transaction_id"].is_unique


`pd.to_numeric(..., errors="coerce")` converts unusable numeric text to missing; it does **not** decide whether to drop or replace that row. Inspect the original `units` entries first. Make new parsed columns so the raw values remain visible. Parse `transaction_date` separately with `pd.to_datetime(..., errors="coerce")`.

**Prediction:** If a missing price were filled with zero, which business meaning would be asserted without evidence?


In [ ]:
prepared = working.copy()
prepared["units_num"] = pd.to_numeric(prepared["units"], errors="coerce")
print("Raw units that did not become numeric:")
display(prepared.loc[prepared["units_num"].isna(), ["transaction_id", "units"]])

# TODO 4: Add price_num, discount_num and parsed_date using safe conversion.
# Keep the originals for inspection. The intended price is USD per unit;
# the discount is a fraction in [0, 1].
required_parsed = ["price_num", "discount_num", "parsed_date"]
if any(name not in prepared for name in required_parsed):
    raise NotImplementedError("Add the three parsed columns before quality checks.")
print(prepared[["units_num", "price_num", "discount_num", "parsed_date"]].isna().sum())


### Make a traceable quality audit

Count **separate reasons** on the deduplicated rows: Cancelled status; missing/nonpositive numeric units; missing/negative price; out-of-range/missing discount; and an unparseable date. Do not silently drop rows yet. Check whether one row has more than one reason. The unmatched hub reason becomes visible after the lookup merge.

Hint: `.isna()`, comparisons such as `.le(0)`, and `.between(0, 1)` each return a Boolean Series. Use `|` for **or** and `~` to invert a condition.


In [ ]:
# TODO 5: Populate one Boolean column per reason, aligned to prepared.index.
quality = pd.DataFrame(index=prepared.index)
quality["cancelled"] = prepared["status"].eq("Cancelled")  # model one reason
# Add: bad_units, bad_price, bad_discount, bad_date.
required_reasons = ["cancelled", "bad_units", "bad_price", "bad_discount", "bad_date"]
if any(name not in quality for name in required_reasons):
    raise NotImplementedError("Complete the four other quality reason columns.")
print("Quality reasons (a row may need inspection if reasons overlap):")
print(quality.sum())
print("Rows with more than one reason:", int(quality.sum(axis=1).gt(1).sum()))


## 4 · Join lookup information, then select eligible transactions

`concat` stacks batches of the **same columns**. A key-based `merge` adds reference columns **across** rows using `hub_code`, like a SQL join or spreadsheet lookup. Check the lookup key's uniqueness before merging: duplicate reference keys can multiply transactions. Use a **left** merge to keep an unmatched transaction inspectable; an inner merge would discard it. `.join()` is another pandas join method, often using indexes, but the explicit-key merge fits this table.


In [ ]:
assert hubs["hub_code"].notna().all() and hubs["hub_code"].is_unique
# TODO 6: Left-merge prepared with hubs on hub_code using
# validate="many_to_one" and indicator=True.
joined = None
if joined is None:
    raise NotImplementedError("Create the validated left merge on hub_code.")
assert len(joined) == len(prepared), "The join multiplied or removed transactions."
print("Join results:", joined["_merge"].value_counts())
display(joined.loc[joined["_merge"].eq("left_only"),
                   ["transaction_id", "hub_code", "hub_name", "region"]])


Now make the **eligible** reporting table. Apply *all* rules together: one copy of each transaction, Fulfilled, positive usable units, present nonnegative unit price, discount from 0 through 1, parsed date and matched hub. Do not remove bad records from `raw` or `working`. Make `.copy()` after selection so a derived column does not ambiguously mutate a view.

The value formula is vectorised column arithmetic, like a spreadsheet formula applied to many rows. For example, a `0.10` discount means multiply the pre-discount amount by `0.90`. Think about whether rounding to cents is appropriate for this constructed USD example.


In [ ]:
# TODO 7: Build the full eligibility condition and a .copy() of the matching rows.
eligible = None
if eligible is None:
    raise NotImplementedError("Filter to unique, validated, matched Fulfilled rows.")

# TODO 8: Derive net_order_value in USD from the three numeric columns.
if "net_order_value" not in eligible:
    raise NotImplementedError("Create net_order_value from units, price and discount.")
assert eligible["transaction_id"].is_unique
assert eligible["net_order_value"].notna().all() and eligible["net_order_value"].ge(0).all()
print("Deduplicated / eligible / excluded:", len(working), len(eligible), len(working) - len(eligible))
print("Matched hub check:", eligible["_merge"].value_counts())
# Reconcile the excluded count with your quality reasons and unmatched-hub count.
# If reason counts overlap, inspect those rows before summing the reasons.


### Guided reconciliation before the final practice

`groupby` splits the eligible table by a business label and recombines an aggregate. Here `size` counts rows; `count` counts nonmissing values of a selected column, so they need not always match. First summarise transaction count and total value by hub, then by hub/category. Check that both grouped sums equal the eligible row-level sum **and** that their transaction counts equal the eligible row count.

Hint: `eligible.groupby(["hub_name"], as_index=False).agg(transactions=("transaction_id", "size"), ...)` supplies one named aggregation. Supply the value aggregation and then try two grouping labels.


In [ ]:
# TODO 9: Create by_hub and by_hub_category with transaction counts and
# net_order_value_usd sums; round displayed USD values to two decimals.
by_hub = None
by_hub_category = None
if by_hub is None or by_hub_category is None:
    raise NotImplementedError("Complete the two groupby summaries.")
display(by_hub, by_hub_category)
row_total = eligible["net_order_value"].sum()
hub_total = by_hub["net_order_value_usd"].sum()
hub_category_total = by_hub_category["net_order_value_usd"].sum()
print("Row / hub / hub-category USD totals:", row_total, hub_total, hub_category_total)
assert np.isclose(row_total, hub_total) and np.isclose(row_total, hub_category_total)
assert by_hub["transactions"].sum() == by_hub_category["transactions"].sum() == len(eligible)


## 5 · Exploratory visual evidence

Use this loop: **question → row grain and quality → valid population → numerical summary → suitable graphic → supported finding → limitation**. A hub total compares groups; a histogram asks how individual eligible order values are distributed. Make **one histogram** of `net_order_value` in USD for eligible Fulfilled transactions across 1–14 September 2026. This is distinct from the grouped total table. Name the population and currency on the plot. Outlying values are prompts to inspect, not automatic errors or causal evidence.

Optional after the required histogram: draw a simple bar chart from the already summed `by_hub` table. Do not let a Seaborn bar silently show a mean when the question asks for a total.


In [ ]:
# TODO 10: Use sns.histplot on eligible order values, then set a precise
# title and both axis labels. End with plt.show().
# Hint: sns.histplot(data=eligible, x="net_order_value", bins=12,
#                    ax=ax, color="steelblue")
fig, ax = plt.subplots(figsize=(7, 4))
raise NotImplementedError("Draw and label the eligible-value histogram.")


**Write below the graphic:** What does its horizontal axis represent? Give one supported observation about the distribution and one limitation of this small synthetic sample. The grouped table may support a separate hub comparison, but neither visual supports a causal story about hub performance.

> Your local finding and limitation: …


## 6 · A public API is another way data arrives

`Python requests client → HTTPS GET + query parameters → HTTP status → JSON body → observation records → pandas DataFrame → plot`

An **endpoint** identifies the requested resource; parameters narrow the result. **GET** reads data here. **POST** commonly submits data, but we will not send a POST in this session. A `2xx` status suggests request success; `4xx` can signal a client/authentication problem; `5xx` a server problem. Check status *and* response structure. JSON arrays and objects become Python lists and dictionaries. Some APIs require keys, but this selected World Bank Indicators endpoint is documented as public and needs no key.

**Consume versus expose:** Here our notebook *consumes* an existing World Bank endpoint. A later application might *expose* its own endpoint with a framework such as FastAPI. No server or model deployment is needed here.

The separate API question is: **How did the World Bank's reported total population for India change from 2014 through 2023?** Its values are midyear estimates in **persons**, not a retailer metric and not evidence of why population changed.


In [ ]:
API_URL = "https://api.worldbank.org/v2/country/IND/indicator/SP.POP.TOTL"
API_PARAMS = {"format": "json", "date": "2014:2023", "per_page": 50}
print("Endpoint:", API_URL)
print("Query parameters:", API_PARAMS)


### Live-first attempt · learner-owned final practice

Complete one real `requests.get` call using `API_URL`, `API_PARAMS` and a bounded `timeout=(10, 25)`. Print the status, call `raise_for_status()` **before** `.json()`, and inspect the top-level JSON shape. A failed network/HTTP/JSON/shape attempt must remain visible. An unfinished TODO is a coding task, not a reason to pretend that the snapshot was fetched live.

Hint 1: `requests.get(API_URL, params=API_PARAMS, timeout=(10, 25))`. Hint 2: `response.json()` decodes JSON but does not by itself prove HTTP success.


In [ ]:
def fetch_live_payload():
    # TODO 11: Replace None with the actual one-request GET call.
    response = None
    if response is None:
        raise NotImplementedError("Write the requests.get(...) call first.")
    print("HTTP status:", response.status_code)
    response.raise_for_status()
    return response.json()


live_error = None
api_payload = None
source_mode = None
try:
    api_payload = fetch_live_payload()
    if not isinstance(api_payload, list) or len(api_payload) != 2:
        raise ValueError("Expected [metadata, observation records] JSON array")
    source_mode = "live World Bank GET"
    print("Live response top-level types:", [type(part).__name__ for part in api_payload])
except (requests.exceptions.RequestException, ValueError) as exc:
    live_error = exc
    api_payload = None
    print("LIVE REQUEST FAILED:", type(exc).__name__, str(exc))
    print("A separate offline choice is available in the next cell.")


### Deliberate offline choice if live retrieval failed

If the live attempt printed an error, first read it. Then set `USE_OFFLINE = True` below and rerun this cell. It loads the **actual World Bank response captured 17 October 2026, 09:11 UTC**; its response metadata says `lastupdated=2026-10-08`. If the live call worked, leave `USE_OFFLINE = False`. Do not describe an offline load as a live request. The same downstream JSON inspection and plot cells apply to either source.


In [ ]:
USE_OFFLINE = False  # Change deliberately to True after a visible live failure.
if USE_OFFLINE:
    with (DATA_DIR / "S12_WorldBank_Population_Snapshot.json").open(
        "r", encoding="utf-8"
    ) as stream:
        api_payload = json.load(stream)
    source_mode = "offline snapshot captured 2026-10-17 09:11 UTC"
    print("OFFLINE SNAPSHOT selected; no HTTP request in this cell.")
elif api_payload is None:
    raise RuntimeError(
        "No usable live response. Read the error above, then set USE_OFFLINE=True "
        "to load the separately labelled snapshot."
    )
print("Current source:", source_mode)


### Inspect structure before flattening

The API returns **two elements**: `[0]` is metadata, `[1]` is the observation list. Do not chart the metadata as a population row. Check that the first page contains the full result (`page`, `pages`, `total`, `per_page`) and inspect one record's keys and nested `country`/`indicator`. Check the year set, duplicates, null values and `IND` / `SP.POP.TOTL` identity before using the series. A later live response may contain revised values; do not fill missing observations with fabricated numbers.

Hint: `isinstance(api_payload, list)`, `metadata, observations = api_payload`, `pd.DataFrame(observations)[["date", "value"]]` for a first look. Keep explicit error messages if the structure is not usable.


In [ ]:
# TODO 12: Validate the two-element structure, unpack metadata and records,
# inspect pagination and one observation, then check identity/year/value quality.
metadata = None
observations = None
if metadata is None or observations is None:
    raise NotImplementedError("Inspect metadata and observation records first.")
print("Metadata:", metadata)
print("First observation keys:", list(observations[0]))


### Make a chronological pandas table

Select the observation `date` and `value` fields, rename them to `year` and `population`, convert year to numeric, inspect missingness, and sort from 2014 upward. `value` represents **persons** even though each record's `unit` string is blank. If a later live response contains a missing value, report it and omit only that year from the plotted table rather than inventing a population count.


In [ ]:
# TODO 13: Build year_population from observations; parse and inspect both
# columns, handle genuine nulls explicitly, and sort by ascending year.
year_population = None
if year_population is None:
    raise NotImplementedError("Create and inspect the year/persons DataFrame.")
display(year_population)
print("Current source:", source_mode)


### Draw and interpret a source-labelled line

Use numeric year on the horizontal axis and population **(persons)** on the vertical axis. Show 2014–2023 in the title, retain the current `source_mode`, and cite World Bank, World Development Indicators, `SP.POP.TOTL`. For an offline run include the snapshot retrieval date; for a live run identify it as live. Underlying providers and CC BY 4.0 terms are described in the [indicator metadata](https://databank.worldbank.org/metadataglossary/world-development-indicators/series/SP.POP.TOTL) and [World Bank dataset terms](https://www.worldbank.org/ext/en/legal/terms-conditions/datasets). Attribution does not imply World Bank endorsement.


In [ ]:
# TODO 14: Plot year_population in year order, label axes/title, and add a
# compact source credit that says live or dated offline snapshot. Call plt.show().
fig, ax = plt.subplots(figsize=(8, 4))
raise NotImplementedError("Complete the year-versus-population line plot.")


## 7 · Interpret, check, save

Write short answers in your own words:

1. Which raw transaction quality issue required **exclusion rather than zero filling**, and why? What happened to the original raw record?
2. Why did the left hub merge preserve a record that an inner merge would lose? Did the reference lookup multiply rows?
3. What does your eligible-value visual or hub summary show, and what can this synthetic sample **not** establish?
4. What changed across the API years you actually received? Why is the line not a causal explanation or an exact census count?
5. Was your API result **live** or the **17 October 2026 offline snapshot**? Record that beside your chart.

> Your answers: …

**Before saving:** rerun completed cells from the top in order; ensure the current `USE_OFFLINE` setting accurately describes your source; check grouped counts/sums and chart labels; save the `.ipynb`. A fresh-kernel *Run All* is meaningful only after every TODO is completed. Optional extra: chart the already aggregated hub totals with honest USD labels.
